# Python 클래스 파일별 4 · 11.ipynb 클래스 종합 실습 · 상품 · 장바구니 · 할인 정책

**한 줄 요약** · 상품 · 장바구니 · 할인 정책 · 고객을 클래스로 연결하며 검증 · 컴포지션 · 다형성을 복습하고 새 정책을 추가

- **원본** · `2.ipynb/01_강의노트/11.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/02_Python클래스` 전체 정리 3장 (특수 메서드) · 4장 (상속) · 파일별 1 ~ 3

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. `@property` · setter로 잘못된 가격 · 할인율을 객체가 스스로 막게 만든다
2. 장바구니가 상품 객체를 가지는 관계(컴포지션)와 `__len__` · `__bool__`을 설명한다
3. 할인 정책을 부모 클래스로 묶고 `calculate()`를 오버라이딩해 다형성을 만든다
4. 고객 코드를 고치지 않고 새 할인 정책 클래스를 추가한다

## 어디에 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">실제 상황</th>
<th style="text-align: left;">이 권에서 배우는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">모델이 층 객체를 여러 개 가질 때</td>
<td style="text-align: left;">컴포지션 — <code>Cart</code>가 <code>Product</code>를 가짐 (1-2)</td>
</tr>
<tr>
<td style="text-align: left;">옵티마이저 · 손실 함수를 바꿔 끼울 때</td>
<td style="text-align: left;">다형성 — 같은 <code>calculate()</code> 호출 (2-1 · 2-2)</td>
</tr>
<tr>
<td style="text-align: left;">잘못된 설정값을 막을 때</td>
<td style="text-align: left;"><code>@property</code> setter 검증 (1-1)</td>
</tr>
<tr>
<td style="text-align: left;">새 기능을 기존 코드 수정 없이 붙일 때</td>
<td style="text-align: left;">부모 클래스를 상속한 새 정책 (3장)</td>
</tr>
</tbody>
</table>

## 학습 지도

<img src="images/pfc4_map.png" alt="11.ipynb 학습 지도" width="763">

**예상 학습 시간** · 개념 50분 + 실습 20분

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">장</th>
<th style="text-align: left;">제목</th>
<th style="text-align: left;">이 장의 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">0</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">실행 환경 · 원본 구성</td>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">상품과 장바구니</td>
<td style="text-align: left;">상품은 자기 값을 어떻게 지키고, 장바구니는 상품을 어떻게 가지는가?</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">할인 정책과 고객</td>
<td style="text-align: left;">여러 할인 정책을 고객이 같은 방식으로 쓰게 하려면?</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">종합 실습</td>
<td style="text-align: left;">잘못된 값은 객체가 막고, 새 할인 정책은 고객 코드를 고치지 않고 붙이려면?</td>
</tr>
<tr>
<td style="text-align: left;">—</td>
<td style="text-align: left;">마무리 · 부록</td>
<td style="text-align: left;">오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td>
</tr>
</tbody>
</table>

---
## 0. 준비

이 노트북은 라이브러리 없이 파이썬 기본 문법만 씀. **셀은 위에서부터 차례로 실행**(1-1의 `Product`부터 클래스가 이어짐).

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">항목</th>
<th style="text-align: left;">설정</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">라이브러리</td>
<td style="text-align: left;">없음</td>
</tr>
<tr>
<td style="text-align: left;">seed · 장치</td>
<td style="text-align: left;">필요 없음</td>
</tr>
<tr>
<td style="text-align: left;">데이터</td>
<td style="text-align: left;">원본의 상품 2개(키보드 35,000원 · 마우스 18,000원)</td>
</tr>
</tbody>
</table>

In [1]:
# 실행 환경 확인: 이 노트북은 파이썬 기본 문법만 씀
import sys  # 파이썬 버전 확인용

print("Python:", sys.version.split()[0])  # 3.10 이상이면 됨

Python: 3.14.6


### 원본 구성

- 원본 주제 · 상품 · 장바구니 · 할인 정책 · 고객을 함께 설계하는 클래스 종합 실습

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본 절</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">이 노트북</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">상품: 속성 검증과 문자열 표현</td>
<td style="text-align: left;">1-1</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">장바구니: 상품 객체를 여러 개 가짐</td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">할인 정책: 같은 <code>calculate()</code>가 정책마다 다르게 동작</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">4 ~ 6</td>
<td style="text-align: left;">고객 · 전체 흐름 · 객체 독립성</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">7 ~ 8</td>
<td style="text-align: left;">잘못된 값 막기 · 핵심 정리</td>
<td style="text-align: left;">3</td>
</tr>
</tbody>
</table>

- 원본 노트북에는 실행 출력이 저장돼 있지 않음 → 이 노트북에서 모두 실행해 출력을 확인

**원본 힌트**

- Product는 자기 가격을 검증 · Cart는 Product 객체를 가짐 → 컴포지션
- 할인 정책들은 DiscountPolicy를 상속 · 같은 `calculate()`가 정책마다 다르게 → 오버라이딩과 다형성
- 새 할인 정책은 Customer를 고치지 않고 새 클래스로 추가할 수 있음

---
## 1. 상품과 장바구니

> **이 장의 흐름**
>
> - 원본 1절: 가격을 검사하는 Product
> - 원본 2절: Product 객체 여러 개를 가지는 Cart

<img src="images/pfc4_compose.png" alt="고객 → 장바구니 → 상품 객체 (가지는 관계)" width="763">

**파이프라인 위치** · ④ 모델 — 객체 안에 객체

### 1-1. `A` 상품 · `@property`로 값 검증 · `@price.setter` · `__repr__` <sub>원본 1절</sub>

> 🎯 **`keyboard.price = "1000"`처럼 잘못된 가격을 넣으면 어떻게 막는가?**

💡 **쉽게 말하면**

- `@property`는 **속성처럼 보이는 메서드**.
- 값을 넣을 때마다 **setter가 먼저 검사**해서 틀리면 에러로 거절.

**실행 흐름**

1. Product 정의 (getter · setter)
2. 상품 2개 생성
3. str · repr 출력
4. 문자열 가격 넣어 보기

**먼저 예측** · `keyboard.price = "1000"` 뒤 가격은 바뀔까? — 답은 코드 실행 결과 아래에

In [2]:
class Product:                        # 상품 설계도
    def __init__(self, name, price):  # 이름 · 가격
        self.name = name              # 상품 이름
        self.price = price            # 아래 setter를 거쳐 검사 후 저장

    @property                # 속성처럼 읽는 메서드
    def price(self):         # product.price로 읽을 때 실행
        return self.__price  # 숨긴 실제 값

    @price.setter                                                  # product.price = 값 할 때 실행
    def price(self, value):                                        # 저장 전에 검사
        if not isinstance(value, int) or isinstance(value, bool):  # 정수만 (True · False 제외)
            raise TypeError('가격은 정수여야 합니다.')             # 자료형이 틀림
        if value < 0:                                              # 음수 가격
            raise ValueError('가격은 0 이상이어야 합니다.')        # 값이 틀림
        self.__price = value                                       # 통과하면 저장

    def __str__(self):                           # print용
        return f'{self.name}: {self.price:,}원'  # 천 단위 쉼표

    def __repr__(self):                                   # 리스트 안 · 디버깅용
        return f'Product({self.name!r}, {self.price!r})'  # 다시 만들 수 있는 모양


keyboard = Product('키보드', 35_000)   # 상품 1 (35_000 = 35000)
mouse = Product('마우스', 18_000)      # 상품 2
print(keyboard)                        # __str__
print(repr(mouse))                     # __repr__
try:
    keyboard.price = "1000"            # 문자열 가격으로 바꾸기
except TypeError as e:                 # setter가 막음
    print("TypeError:", e)             # 에러 문장
print("가격 그대로:", keyboard.price)  # 저장되지 않음

키보드: 35,000원
Product('마우스', 18000)
TypeError: 가격은 정수여야 합니다.
가격 그대로: 35000


**결과** · 예측 정답은 아니요 — setter가 TypeError로 막아 35000 그대로

- `__str__`은 사용자용, `__repr__`은 다시 만들 수 있는 모양
  - `키보드: 35,000원`
  - `Product('마우스', 18000)`
- `TypeError: 가격은 정수여야 합니다.` — setter가 문자열 가격을 거절
- `가격 그대로: 35000` — 검사에 실패하면 저장되지 않음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>self.price = 값</code> (일반 속성)</td>
<td style="text-align: left;">아무 값이나 저장</td>
<td style="text-align: left;">검사가 필요 없을 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>@property</code> + setter</td>
<td style="text-align: left;">저장 전에 <strong>검사</strong></td>
<td style="text-align: left;">가격 · 할인율처럼 범위가 있는 값</td>
</tr>
<tr>
<td style="text-align: left;"><code>self.__price</code></td>
<td style="text-align: left;">이름을 바꿔 숨김 (밖에서 직접 접근 어려움)</td>
<td style="text-align: left;">setter를 거치게 할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - setter 안에서 `self.price = value`로 쓰면 setter를 다시 불러 무한 반복 → 실제 값은 `self.__price`에

> ✅ **한 줄 정리** · `@property`와 setter를 쓰면 값을 넣을 때마다 검사해서 잘못된 값을 객체가 스스로 막는다.

**딥러닝 코드에서는**

- `tensor.T` · `df.shape`처럼 괄호 없이 읽지만 실제로는 메서드가 실행되는 property

### 1-2. `A` 장바구니 · 컴포지션 · `list(products)` · `__len__` · `__bool__` <sub>원본 2절</sub>

> 🎯 **장바구니는 상품을 어떻게 가지고, `len()` · `if cart:`는 어떻게 동작하는가?**

💡 **쉽게 말하면**

- 장바구니는 **상품 객체를 담은 리스트**를 가짐 (가지는 관계 = 컴포지션).
- `__len__`은 `len()`, `__bool__`은 `if`에서 참 · 거짓.

**실행 흐름**

1. Cart 정의
2. 리스트를 복사해 보관
3. 밖의 리스트 바꿔 보기
4. len · bool · 잘못된 추가

**먼저 예측** · `Cart(items)` 뒤 `items`에 모니터를 추가하면 장바구니 상품 수는? — 답은 코드 실행 결과 아래에

In [3]:
class Cart:                             # 장바구니: 상품 객체를 여러 개 가짐 (컴포지션)
    def __init__(self, products=None):  # 기본값 None (파일별 5의 함정 피하기)
        self.products = [] if products is None else list(products)  # 복사해서 보관

    def add(self, product):                                        # 상품 추가
        if not isinstance(product, Product):                       # Product만 허용
            raise TypeError('Product 객체만 추가할 수 있습니다.')  # 다른 것은 거절
        self.products.append(product)                              # 목록에 추가

    def total_price(self):                                      # 총액
        return sum(product.price for product in self.products)  # 상품 가격 합

    def __len__(self):             # len(cart)
        return len(self.products)  # 상품 개수

    def __bool__(self):       # if cart: 에서 참 · 거짓
        return len(self) > 0  # 비었으면 False

    def __str__(self):                                                # print용
        return f'상품 {len(self)}개 / 총액 {self.total_price():,}원'  # 요약


items = [keyboard, mouse]                                           # 밖에서 만든 리스트
demo_cart = Cart(items)                                             # 리스트를 복사해 보관
items.append(Product('모니터', 200_000))                            # 밖의 리스트를 바꿔도
print(demo_cart)                                                    # 장바구니는 그대로 2개
print("len:", len(demo_cart), "/ 빈 장바구니는 참?", bool(Cart()))  # __len__ · __bool__
try:
    demo_cart.add("키보드")                                         # 문자열을 상품처럼 추가
except TypeError as e:                                              # add()가 막음
    print("TypeError:", e)                                          # 에러 문장

상품 2개 / 총액 53,000원
len: 2 / 빈 장바구니는 참? False
TypeError: Product 객체만 추가할 수 있습니다.


**결과** · 예측 정답은 2개 그대로 — `list(products)`로 복사해 보관

- `상품 2개 / 총액 53,000원` — 35,000 + 18,000 · 밖의 리스트 변경과 무관
- `len: 2 / 빈 장바구니는 참? False` — `__len__` · `__bool__`이 동작
- `TypeError: Product 객체만 추가할 수 있습니다.` — add()가 자료형을 검사

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">상속 (is-a)</td>
<td style="text-align: left;">자식은 부모의 <strong>한 종류</strong></td>
<td style="text-align: left;"><code>RateDiscount</code>는 할인 정책 (2장)</td>
</tr>
<tr>
<td style="text-align: left;">컴포지션 (has-a)</td>
<td style="text-align: left;">객체가 다른 객체를 <strong>가짐</strong></td>
<td style="text-align: left;">장바구니는 상품을 가짐</td>
</tr>
<tr>
<td style="text-align: left;"><code>list(products)</code> 복사</td>
<td style="text-align: left;">밖의 리스트와 분리</td>
<td style="text-align: left;">넘겨받은 리스트를 보관할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `self.products = products`로 그대로 두면 밖에서 리스트를 바꿀 때 장바구니도 바뀜 (파일별 1의 `y = x`)

> ✅ **한 줄 정리** · 다른 객체를 가지는 관계가 컴포지션이며, `__len__` · `__bool__`로 `len()` · `if`에 반응하게 만든다.

**딥러닝 코드에서는**

- `nn.Sequential`이 층 객체들을 가지는 것 · `DataLoader`가 `Dataset`을 가지는 것도 컴포지션

---
## 2. 할인 정책과 고객

> **이 장의 흐름**
>
> - 원본 3절: 부모 DiscountPolicy와 정책 세 개
> - 원본 4 ~ 6절: 정책을 받아 결제하는 Customer · 객체 독립성

<img src="images/pfc4_policy.png" alt="할인 정책 상속과 같은 calculate() 호출" width="763">

**파이프라인 위치** · ④ 모델 · ⑤ 학습 — 바꿔 끼우는 부품

### 2-1. `A` 할인 정책 · 상속과 오버라이딩 · `calculate()` · `NotImplementedError` <sub>원본 3절</sub>

> 🎯 **정책마다 다른 할인을 같은 `calculate()` 이름으로 쓰려면?**

💡 **쉽게 말하면**

- 부모는 **약속만**(`calculate`를 만들 것), 자식마다 **계산 방법**을 채움.
- 부모를 그대로 쓰면 `NotImplementedError`로 알려 줌.

**실행 흐름**

1. 부모 DiscountPolicy
2. 자식 정책 세 개
3. 같은 금액에 정책마다 calculate
4. 부모 직접 호출

**먼저 예측** · 53,000원에 10% 할인을 적용하면? — 답은 코드 실행 결과 아래에

In [4]:
class DiscountPolicy:            # 부모: 할인 정책의 공통 약속
    def calculate(self, price):  # 자식이 반드시 다시 정의할 메서드
        raise NotImplementedError('자식 클래스에서 calculate()를 구현하세요.')  # 부모만으로는 못 씀


class NoDiscount(DiscountPolicy):  # 할인 없음
    def calculate(self, price):    # 오버라이딩
        return price               # 그대로

    def __str__(self):      # 정책 이름
        return '할인 없음'  # 출력용


class RateDiscount(DiscountPolicy):  # 비율 할인
    def __init__(self, rate):        # 할인율 (0 ~ 1)
        self.rate = rate             # setter로 검사 후 저장

    @property               # 1-1과 같은 방식
    def rate(self):         # 읽기
        return self.__rate  # 숨긴 실제 값

    @rate.setter                                                            # 넣을 때 검사
    def rate(self, value):                                                  # 저장 전에 검사
        if not isinstance(value, (int, float)) or isinstance(value, bool):  # 숫자만
            raise TypeError('할인율은 숫자여야 합니다.')                    # 자료형 틀림
        if not 0 <= value <= 1:                                             # 범위 밖
            raise ValueError('할인율은 0 이상 1 이하여야 합니다.')          # 값 틀림
        self.__rate = value                                                 # 통과하면 저장

    def calculate(self, price):                # 오버라이딩
        return round(price * (1 - self.rate))  # 비율만큼 깎고 반올림

    def __str__(self):                  # 정책 이름
        return f'{self.rate:.0%} 할인'  # 0.1 → 10%


class AmountDiscount(DiscountPolicy):                                # 금액 할인
    def __init__(self, amount):                                      # 할인 금액
        if not isinstance(amount, int) or isinstance(amount, bool):  # 정수만
            raise TypeError('할인 금액은 정수여야 합니다.')          # 자료형 틀림
        if amount < 0:                                               # 음수
            raise ValueError('할인 금액은 0 이상이어야 합니다.')     # 값 틀림
        self.amount = amount                                         # 저장

    def calculate(self, price):             # 오버라이딩
        return max(0, price - self.amount)  # 0원 아래로는 안 내려감

    def __str__(self):                    # 정책 이름
        return f'{self.amount:,}원 할인'  # 5000 → 5,000원


policies = [NoDiscount(), RateDiscount(0.1), AmountDiscount(5_000)]  # 종류가 다른 정책
for policy in policies:                                              # 종류를 묻지 않고
    print(f"{policy}: {policy.calculate(53_000):,}원")               # 같은 calculate() 호출
try:
    DiscountPolicy().calculate(1000)                                 # 부모를 그대로 사용
except NotImplementedError as e:                                     # 구현이 없음
    print("NotImplementedError:", e)                                 # 에러 문장

할인 없음: 53,000원
10% 할인: 47,700원
5,000원 할인: 48,000원
NotImplementedError: 자식 클래스에서 calculate()를 구현하세요.


**결과** · 예측 정답은 `47,700원` — 53,000 × 0.9

- 같은 `calculate()`가 정책마다 다른 결과 (다형성)
  - `할인 없음: 53,000원`
  - `10% 할인: 47,700원`
  - `5,000원 할인: 48,000원`
- `NotImplementedError: 자식 클래스에서 calculate()를 구현하세요.` — 부모는 약속만 하고 계산은 자식이

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">부모 <code>DiscountPolicy</code></td>
<td style="text-align: left;"><code>calculate()</code> 약속만 · 직접 쓰면 에러</td>
<td style="text-align: left;">공통 이름을 정할 때</td>
</tr>
<tr>
<td style="text-align: left;">자식 정책 3개</td>
<td style="text-align: left;">같은 이름으로 <strong>각자 계산</strong> (오버라이딩)</td>
<td style="text-align: left;">할인 방식마다</td>
</tr>
<tr>
<td style="text-align: left;"><code>if kind == "rate":</code> 분기</td>
<td style="text-align: left;">종류마다 if로 나눔</td>
<td style="text-align: left;">쓰지 않기 (정책이 늘면 고칠 곳이 늘어남)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 자식이 `calculate()` 이름을 틀리게 쓰면(`calc`) 오버라이딩이 안 돼 부모의 NotImplementedError가 남

> ✅ **한 줄 정리** · 부모 클래스로 `calculate()` 이름을 약속하고 자식마다 다시 정의하면, 같은 호출이 정책마다 다르게 동작한다.

**딥러닝 코드에서는**

- `nn.Module`의 `forward()`처럼 부모가 이름을 정하고 자식이 내용을 채우는 구조

### 2-2. `A` 고객 · 결제와 객체 독립성 · `checkout()` · `is` <sub>원본 4 ~ 6절</sub>

> 🎯 **고객은 어떤 정책이 와도 같은 코드로 결제할 수 있는가?**

💡 **쉽게 말하면**

- 고객은 **정책의 종류를 묻지 않고** `calculate()`만 부름.
- 고객마다 장바구니는 **따로** (None 기본값 + 안에서 생성).

**실행 흐름**

1. Customer 정의
2. 정책 3개로 결제
3. 다른 고객 만들기
4. 장바구니 공유 확인 · 빈 장바구니

**먼저 예측** · 새 고객 `지수`의 장바구니는 `민수`의 장바구니와 같은 객체일까? — 답은 코드 실행 결과 아래에

In [5]:
class Customer:                                       # 고객: 장바구니를 가지고 정책을 받아 결제
    def __init__(self, name, cart=None):              # 장바구니는 선택
        self.name = name                              # 고객 이름
        self.cart = Cart() if cart is None else cart  # 없으면 고객마다 새 장바구니

    def add_to_cart(self, product):  # 상품 담기
        self.cart.add(product)       # 장바구니에 맡김 (위임)

    def checkout(self, discount_policy):                                  # 결제
        if not isinstance(discount_policy, DiscountPolicy):               # 정책 객체만 허용
            raise TypeError('DiscountPolicy 객체가 필요합니다.')          # 잘못된 인자
        if not self.cart:                                                 # Cart.__bool__ 사용
            return f'{self.name}: 장바구니가 비어 있습니다.'              # 빈 장바구니
        final_price = discount_policy.calculate(self.cart.total_price())  # 어떤 정책이든 같은 호출
        return f'{self.name} / {discount_policy} / 결제 금액 {final_price:,}원'  # 결과 문장


customer = Customer('민수', demo_cart)                           # 1-2의 장바구니를 가진 고객
for policy in policies:                                          # 2-1의 정책 세 개
    print(customer.checkout(policy))                             # 고객 코드는 하나
another_customer = Customer('지수')                              # 장바구니 없이 만든 고객
print("장바구니 공유?", customer.cart is another_customer.cart)  # 객체 독립성
print(another_customer.checkout(NoDiscount()))                   # 빈 장바구니

민수 / 할인 없음 / 결제 금액 53,000원
민수 / 10% 할인 / 결제 금액 47,700원
민수 / 5,000원 할인 / 결제 금액 48,000원
장바구니 공유? False
지수: 장바구니가 비어 있습니다.


**결과** · 예측 정답은 아니요 — `None`이면 고객마다 새 `Cart()`

- 정책만 바꿔 끼워도 고객 코드는 그대로
  - `민수 / 할인 없음 / 결제 금액 53,000원`
  - `민수 / 10% 할인 / 결제 금액 47,700원`
  - `민수 / 5,000원 할인 / 결제 금액 48,000원`
- `장바구니 공유? False` — 고객마다 자기 장바구니
- `지수: 장바구니가 비어 있습니다.` — `Cart.__bool__`이 False

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>discount_policy.calculate(...)</code></td>
<td style="text-align: left;">종류를 묻지 않고 같은 호출</td>
<td style="text-align: left;">정책을 바꿔 끼울 때 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;"><code>self.cart.add(...)</code></td>
<td style="text-align: left;">장바구니에 일을 맡김 (위임)</td>
<td style="text-align: left;">가진 객체의 기능을 쓸 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>cart=None</code> + 안에서 생성</td>
<td style="text-align: left;">고객마다 새 장바구니</td>
<td style="text-align: left;">바뀌는 객체를 기본값으로 쓸 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `def __init__(self, name, cart=Cart())`로 쓰면 모든 고객이 같은 장바구니를 공유 (파일별 5의 기본값 함정)

> ✅ **한 줄 정리** · 고객은 정책의 종류를 묻지 않고 `calculate()`만 불러 결제하므로, 정책을 바꿔 끼워도 고객 코드는 그대로다.

**딥러닝 코드에서는**

- `train(model, optimizer)`가 SGD · Adam 어떤 옵티마이저든 `optimizer.step()`만 부르는 것과 같음

---
## 3. 종합 실습 · 잘못된 값 막기와 새 정책 추가

> **이 장의 흐름**
>
> - 1 · 2장 문법을 한 번에 사용
> - 원본 7절의 에러 세 가지를 확인하고, 원본 정리의 '새 정책 추가'를 직접 해 봄

<img src="images/pfc4_practice.png" alt="잘못된 값은 막히고, 새 정책은 Customer 수정 없이 연결" width="763">

**파이프라인 위치** · ④ 모델 → ⑤ 학습

> 🎯 **음수 가격 · 1.5 할인율 · 문자열 정책을 넣으면 어떻게 되고, 쿠폰 정책은 어떻게 추가하는가?**

💡 **쉽게 말하면**

- **잘못된 값** · 만드는 순간 · 결제하는 순간에 에러로 거절.
- **새 정책** · `DiscountPolicy`를 물려받아 `calculate()`만 채우면 끝.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 문법</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">잘못된 값 세 가지</td>
<td style="text-align: left;">1-1 setter · 2-1 검사 · 2-2 isinstance</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">새 정책 CouponDiscount</td>
<td style="text-align: left;">2-1 상속 · 오버라이딩</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">Customer 수정 없이 결제</td>
<td style="text-align: left;">2-2 다형성</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">다른 고객 · 다른 금액</td>
<td style="text-align: left;">1-2 컴포지션</td>
</tr>
</tbody>
</table>

**먼저 예측** · 53,000원 장바구니에 '5만 원 이상 7,000원' 쿠폰을 쓰면 결제 금액은? — 답은 코드 실행 결과 아래에

In [6]:
# 1단계: 잘못된 값은 객체가 스스로 막음 (원본 7절)
cases = [lambda: Product('잘못된 상품', -1),     # 음수 가격
         lambda: RateDiscount(1.5),              # 1을 넘는 할인율
         lambda: customer.checkout('10% 할인')]  # 정책 객체가 아닌 문자열
for make in cases:                               # 하나씩 실행
    try:
        make()                                   # 잘못된 값으로 만들거나 결제
    except (TypeError, ValueError) as error:     # 자료형 · 값 에러
        print(type(error).__name__, error)       # 에러 종류와 문장


# 2단계: 새 정책 — DiscountPolicy를 물려받아 calculate()만 채움
class CouponDiscount(DiscountPolicy):                # 일정 금액 이상이면 쿠폰 할인
    def __init__(self, minimum, amount):             # 최소 금액 · 할인 금액
        self.minimum, self.amount = minimum, amount  # 조건 저장

    def calculate(self, price):                                         # 오버라이딩
        return price - self.amount if price >= self.minimum else price  # 조건을 넘을 때만

    def __str__(self):                                            # 정책 이름
        return f'{self.minimum:,}원 이상 {self.amount:,}원 쿠폰'  # 출력용


# 3 · 4단계: Customer는 한 줄도 고치지 않고 새 정책으로 결제
coupon = CouponDiscount(50_000, 7_000)    # 5만 원 이상 7천 원 할인
print(customer.checkout(coupon))          # 53,000원 → 조건 충족
another_customer.add_to_cart(mouse)       # 지수는 마우스만
print(another_customer.checkout(coupon))  # 18,000원 → 조건 미달

ValueError 가격은 0 이상이어야 합니다.
ValueError 할인율은 0 이상 1 이하여야 합니다.
TypeError DiscountPolicy 객체가 필요합니다.
민수 / 50,000원 이상 7,000원 쿠폰 / 결제 금액 46,000원
지수 / 50,000원 이상 7,000원 쿠폰 / 결제 금액 18,000원


**결과** · 예측 정답은 `46,000원` — 53,000 - 7,000

- 만드는 순간 · 결제하는 순간에 객체가 막음
  - `ValueError 가격은 0 이상이어야 합니다.`
  - `ValueError 할인율은 0 이상 1 이하여야 합니다.`
  - `TypeError DiscountPolicy 객체가 필요합니다.`
- `민수 / 50,000원 이상 7,000원 쿠폰 / 결제 금액 46,000원` — 새 정책도 Customer 수정 없이 동작
- `지수 / 50,000원 이상 7,000원 쿠폰 / 결제 금액 18,000원` — 조건 미달이라 할인 없음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">setter · <code>__init__</code> 검사</td>
<td style="text-align: left;">만드는 순간 막음</td>
<td style="text-align: left;">잘못된 값이 들어오면 안 될 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>isinstance</code> 검사</td>
<td style="text-align: left;">엉뚱한 자료형을 막음</td>
<td style="text-align: left;">인자로 객체를 받을 때</td>
</tr>
<tr>
<td style="text-align: left;">새 자식 클래스 추가</td>
<td style="text-align: left;">기존 코드 수정 없음</td>
<td style="text-align: left;">기능을 늘릴 때 (기본 선택)</td>
</tr>
<tr>
<td style="text-align: left;">Customer에 if 추가</td>
<td style="text-align: left;">정책마다 고객 코드를 고침</td>
<td style="text-align: left;">쓰지 않기</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `CouponDiscount`는 검사가 없어 음수 금액도 들어감 → 실제로는 2-1처럼 검사를 넣을 것

> ✅ **한 줄 정리** · 값 검사는 객체 안에 두고, 새 기능은 부모를 물려받은 클래스로 추가하면 기존 코드를 고치지 않아도 된다.

**딥러닝 코드에서는**

- 새 손실 함수를 `nn.Module`로 만들면 학습 루프는 그대로 `loss_fn(pred, y)`를 부름

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">학습 목표</th>
<th style="text-align: left;">할 수 있게 된 것</th>
<th style="text-align: left;">확인 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1. property 검증</td>
<td style="text-align: left;">문자열 가격은 TypeError · 가격 그대로</td>
<td style="text-align: left;">1-1</td>
</tr>
<tr>
<td style="text-align: left;">2. 컴포지션</td>
<td style="text-align: left;">장바구니는 상품 2개 · 53,000원 · 빈 장바구니는 False</td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;">3. 상속 · 다형성</td>
<td style="text-align: left;">같은 calculate() → 53,000 · 47,700 · 48,000</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">4. 고객 결제</td>
<td style="text-align: left;">정책만 바꿔 끼움 · 장바구니는 고객마다 따로</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">5. 검증 · 확장</td>
<td style="text-align: left;">잘못된 값 3가지 차단 · 쿠폰 정책 46,000원</td>
<td style="text-align: left;">3</td>
</tr>
</tbody>
</table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">컴포지션</td>
<td style="text-align: left;">객체가 객체를 가짐 (has-a)</td>
<td style="text-align: left;"><code>nn.Sequential</code> · <code>DataLoader</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">다형성</td>
<td style="text-align: left;">같은 호출, 객체마다 다른 동작</td>
<td style="text-align: left;"><code>optimizer.step()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;"><code>@property</code> setter</td>
<td style="text-align: left;">넣을 때 검사</td>
<td style="text-align: left;">설정값 검증</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;"><code>NotImplementedError</code></td>
<td style="text-align: left;">부모는 약속만</td>
<td style="text-align: left;">자식이 꼭 구현할 메서드</td>
</tr>
</tbody>
</table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">setter는 언제 실행되는가?</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">상속과 컴포지션의 차이는?</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">부모 <code>calculate()</code>를 그대로 부르면?</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">Customer가 정책 종류를 묻지 않아도 되는 이유는?</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">새 정책을 추가할 때 고칠 클래스는?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `Product('모니터', 200_000)`을 만들어 `print`와 `repr`로 출력하기
2. `Cart([keyboard])`의 `len`과 총액 출력하기

In [7]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. `RateDiscount(0.2)`로 53,000원을 결제하면 얼마일까?
2. `AmountDiscount(60_000)`로 53,000원을 계산하면?
3. '2개 이상 사면 3,000원 할인' 정책을 만들려면 `calculate()`에 무엇이 더 필요할까?

In [8]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;"><code>객체.price = 값</code>으로 넣을 때 (생성자 안 포함)</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">~의 한 종류(is-a) vs ~을 가짐(has-a)</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">NotImplementedError</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">모든 정책이 같은 <code>calculate()</code>를 가짐 (다형성)</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">없음 — DiscountPolicy를 상속한 새 클래스만 추가</td>
</tr>
</tbody>
</table>

#### 코드 문제 답

```python
monitor = Product('모니터', 200_000)
print(monitor)  # 모니터: 200,000원
print(repr(monitor))  # Product('모니터', 200000)
c = Cart([keyboard])
print(len(c), c.total_price())  # 1 35000
```

#### 변형 문제 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;"><code>42,400원</code> (53,000 × 0.8)</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;"><code>0원</code> — <code>max(0, ...)</code>로 음수가 되지 않음</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">상품 개수 — <code>calculate(price)</code>만으로는 부족해 <code>calculate(cart)</code>처럼 장바구니를 받도록 약속을 바꿔야 함</td>
</tr>
</tbody>
</table>

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">틀린 내용</th>
<th style="text-align: left;">틀린 이유</th>
<th style="text-align: left;">올바른 원칙</th>
<th style="text-align: left;">재발 방지 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">(예) 새 정책마다 Customer.checkout에 if를 추가함</td>
<td style="text-align: left;">종류마다 분기해야 한다고 착각</td>
<td style="text-align: left;">같은 이름의 메서드를 오버라이딩하면 분기 불필요</td>
<td style="text-align: left;">부모를 상속해 calculate()만 채웠나?</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
</tbody>
</table>

---
## 면접 30초 답변

**Q1. 상속과 컴포지션은 언제 각각 쓰나요?**
> - '~은 ~의 한 종류'면 상속, '~이 ~을 가진다'면 컴포지션을 씁니다.
> - 10% 할인은 할인 정책의 한 종류라 상속하고, 장바구니는 상품을 가지므로 컴포지션입니다.

**Q2. 다형성의 장점은 무엇인가요?**
> - 호출하는 쪽이 객체의 종류를 묻지 않고 같은 메서드만 부르면 됩니다.
> - 그래서 새 정책을 추가해도 고객 코드를 고칠 필요가 없습니다.

**Q3. `@property`는 왜 쓰나요?**
> - 속성처럼 읽고 쓰면서도 값을 넣을 때 검사를 할 수 있습니다.
> - 음수 가격 같은 잘못된 값을 객체가 스스로 막습니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
<th style="text-align: left;">코드에서 사용한 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>@property</code> · <code>@이름.setter</code></td>
<td style="text-align: left;">Python 내장 데코레이터</td>
<td style="text-align: left;">속성처럼 읽기 · 넣을 때 검사</td>
<td style="text-align: left;">1-1 <code>@price.setter</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>__str__</code> · <code>__repr__</code> · <code>__len__</code> · <code>__bool__</code></td>
<td style="text-align: left;">Python 특수 메서드</td>
<td style="text-align: left;">출력 · len() · if 반응</td>
<td style="text-align: left;">1-2 <code>def __bool__(self)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>isinstance()</code> · <code>raise</code></td>
<td style="text-align: left;">Python 내장 함수 · 기본 문법</td>
<td style="text-align: left;">자료형 검사 · 에러 발생</td>
<td style="text-align: left;">1-2 <code>isinstance(product, Product)</code></td>
</tr>
<tr>
<td style="text-align: left;">상속 · 오버라이딩 · <code>NotImplementedError</code></td>
<td style="text-align: left;">Python 기본 문법 · 예외</td>
<td style="text-align: left;">공통 약속 · 자식마다 구현</td>
<td style="text-align: left;">2-1 <code>class RateDiscount(DiscountPolicy)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>sum()</code> · <code>max()</code> · <code>round()</code></td>
<td style="text-align: left;">Python 내장 함수</td>
<td style="text-align: left;">총액 · 하한 · 반올림</td>
<td style="text-align: left;">2-1 <code>max(0, price - self.amount)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>lambda</code> · <code>try</code> · <code>except</code></td>
<td style="text-align: left;">Python 기본 문법</td>
<td style="text-align: left;">잘못된 값 시험</td>
<td style="text-align: left;">3 <code>except (TypeError, ValueError)</code></td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">범위가 있는 값은 setter로 검사</td>
<td style="text-align: left;">setter 안에서 <code>self.price = value</code></td>
</tr>
<tr>
<td style="text-align: left;">가지는 관계는 컴포지션</td>
<td style="text-align: left;">모든 관계를 상속으로</td>
</tr>
<tr>
<td style="text-align: left;">새 기능은 새 자식 클래스로</td>
<td style="text-align: left;">종류마다 if 분기 추가</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">상속 vs 컴포지션</td>
<td style="text-align: left;">is-a vs has-a</td>
<td style="text-align: left;">할인 정책은 상속 · 장바구니는 컴포지션</td>
</tr>
<tr>
<td style="text-align: left;">일반 속성 vs <code>@property</code></td>
<td style="text-align: left;">검사 없음 vs 넣을 때 검사</td>
<td style="text-align: left;">범위가 있으면 property</td>
</tr>
<tr>
<td style="text-align: left;"><code>__len__</code> vs <code>__bool__</code></td>
<td style="text-align: left;">개수 vs 참 · 거짓</td>
<td style="text-align: left;"><code>__bool__</code>이 없으면 <code>__len__</code>으로 판단</td>
</tr>
<tr>
<td style="text-align: left;">부모 메서드 vs 오버라이딩</td>
<td style="text-align: left;">약속 vs 구현</td>
<td style="text-align: left;">부모는 NotImplementedError</td>
</tr>
</tbody>
</table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">언제</th>
<th style="text-align: left;">할 일</th>
<th style="text-align: left;">걸리는 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내일</td>
<td style="text-align: left;">기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td>
<td style="text-align: left;">10분</td>
</tr>
<tr>
<td style="text-align: left;">3일 뒤</td>
<td style="text-align: left;">변형 문제 3개 다시 풀기 · 2-1 결과를 실행 전에 계산</td>
<td style="text-align: left;">15분</td>
</tr>
<tr>
<td style="text-align: left;">7일 뒤</td>
<td style="text-align: left;">3장에 '2개 이상 3,000원 할인' 정책을 직접 설계</td>
<td style="text-align: left;">20분</td>
</tr>
<tr>
<td style="text-align: left;">14일 뒤</td>
<td style="text-align: left;">오답 노트에 남은 문제만 · 전체 정리 5장(nn.Module 종합 실습) 전 ✅ 줄만 읽기</td>
<td style="text-align: left;">10분</td>
</tr>
</tbody>
</table>

---
## 부록

### 이 노트북을 만든 방법

- 1장 · 원본 강의노트(`11.ipynb`) 1 ~ 2절 (상품 · 장바구니)
- 2장 · 3 ~ 6절 (할인 정책 · 고객), 3장 · 7 ~ 8절 검증과 새 정책 추가
- 원본에 저장된 출력이 없어 모든 출력은 이 노트북에서 실행한 결과
- 모든 코드는 실행한 출력과 함께 저장 · 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)

### 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
</tbody>
</table>

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">이 노트북</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1절 상품 · 속성 검증</td>
<td style="text-align: left;">상품 · @property로 값 검증</td>
<td style="text-align: left;">1-1</td>
</tr>
<tr>
<td style="text-align: left;">2절 장바구니</td>
<td style="text-align: left;">장바구니 · 컴포지션</td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;">3절 할인 정책</td>
<td style="text-align: left;">할인 정책 · 상속과 오버라이딩</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">4 ~ 6절 고객 · 흐름 · 독립성</td>
<td style="text-align: left;">고객 · 결제와 객체 독립성</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">7 ~ 8절 잘못된 값 · 정리</td>
<td style="text-align: left;">잘못된 값 막기와 새 정책 추가</td>
<td style="text-align: left;">3</td>
</tr>
</tbody>
</table>

### 전체 코드

1 ~ 3장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

In [9]:
# ---------- 1-1 ----------
class Product:                        # 상품 설계도
    def __init__(self, name, price):  # 이름 · 가격
        self.name = name              # 상품 이름
        self.price = price            # 아래 setter를 거쳐 검사 후 저장

    @property                # 속성처럼 읽는 메서드
    def price(self):         # product.price로 읽을 때 실행
        return self.__price  # 숨긴 실제 값

    @price.setter                                                  # product.price = 값 할 때 실행
    def price(self, value):                                        # 저장 전에 검사
        if not isinstance(value, int) or isinstance(value, bool):  # 정수만 (True · False 제외)
            raise TypeError('가격은 정수여야 합니다.')             # 자료형이 틀림
        if value < 0:                                              # 음수 가격
            raise ValueError('가격은 0 이상이어야 합니다.')        # 값이 틀림
        self.__price = value                                       # 통과하면 저장

    def __str__(self):                           # print용
        return f'{self.name}: {self.price:,}원'  # 천 단위 쉼표

    def __repr__(self):                                   # 리스트 안 · 디버깅용
        return f'Product({self.name!r}, {self.price!r})'  # 다시 만들 수 있는 모양


keyboard = Product('키보드', 35_000)   # 상품 1 (35_000 = 35000)
mouse = Product('마우스', 18_000)      # 상품 2
print(keyboard)                        # __str__
print(repr(mouse))                     # __repr__
try:
    keyboard.price = "1000"            # 문자열 가격으로 바꾸기
except TypeError as e:                 # setter가 막음
    print("TypeError:", e)             # 에러 문장
print("가격 그대로:", keyboard.price)  # 저장되지 않음


# ---------- 1-2 ----------
class Cart:                             # 장바구니: 상품 객체를 여러 개 가짐 (컴포지션)
    def __init__(self, products=None):  # 기본값 None (파일별 5의 함정 피하기)
        self.products = [] if products is None else list(products)  # 복사해서 보관

    def add(self, product):                                        # 상품 추가
        if not isinstance(product, Product):                       # Product만 허용
            raise TypeError('Product 객체만 추가할 수 있습니다.')  # 다른 것은 거절
        self.products.append(product)                              # 목록에 추가

    def total_price(self):                                      # 총액
        return sum(product.price for product in self.products)  # 상품 가격 합

    def __len__(self):             # len(cart)
        return len(self.products)  # 상품 개수

    def __bool__(self):       # if cart: 에서 참 · 거짓
        return len(self) > 0  # 비었으면 False

    def __str__(self):                                                # print용
        return f'상품 {len(self)}개 / 총액 {self.total_price():,}원'  # 요약

키보드: 35,000원
Product('마우스', 18000)
TypeError: 가격은 정수여야 합니다.
가격 그대로: 35000


In [10]:
items = [keyboard, mouse]                                           # 밖에서 만든 리스트
demo_cart = Cart(items)                                             # 리스트를 복사해 보관
items.append(Product('모니터', 200_000))                            # 밖의 리스트를 바꿔도
print(demo_cart)                                                    # 장바구니는 그대로 2개
print("len:", len(demo_cart), "/ 빈 장바구니는 참?", bool(Cart()))  # __len__ · __bool__
try:
    demo_cart.add("키보드")                                         # 문자열을 상품처럼 추가
except TypeError as e:                                              # add()가 막음
    print("TypeError:", e)                                          # 에러 문장

상품 2개 / 총액 53,000원
len: 2 / 빈 장바구니는 참? False
TypeError: Product 객체만 추가할 수 있습니다.


In [11]:
# ---------- 2-1 ----------
class DiscountPolicy:            # 부모: 할인 정책의 공통 약속
    def calculate(self, price):  # 자식이 반드시 다시 정의할 메서드
        raise NotImplementedError('자식 클래스에서 calculate()를 구현하세요.')  # 부모만으로는 못 씀


class NoDiscount(DiscountPolicy):  # 할인 없음
    def calculate(self, price):    # 오버라이딩
        return price               # 그대로

    def __str__(self):      # 정책 이름
        return '할인 없음'  # 출력용


class RateDiscount(DiscountPolicy):  # 비율 할인
    def __init__(self, rate):        # 할인율 (0 ~ 1)
        self.rate = rate             # setter로 검사 후 저장

    @property               # 1-1과 같은 방식
    def rate(self):         # 읽기
        return self.__rate  # 숨긴 실제 값

    @rate.setter                                                            # 넣을 때 검사
    def rate(self, value):                                                  # 저장 전에 검사
        if not isinstance(value, (int, float)) or isinstance(value, bool):  # 숫자만
            raise TypeError('할인율은 숫자여야 합니다.')                    # 자료형 틀림
        if not 0 <= value <= 1:                                             # 범위 밖
            raise ValueError('할인율은 0 이상 1 이하여야 합니다.')          # 값 틀림
        self.__rate = value                                                 # 통과하면 저장

    def calculate(self, price):                # 오버라이딩
        return round(price * (1 - self.rate))  # 비율만큼 깎고 반올림

    def __str__(self):                  # 정책 이름
        return f'{self.rate:.0%} 할인'  # 0.1 → 10%


class AmountDiscount(DiscountPolicy):                                # 금액 할인
    def __init__(self, amount):                                      # 할인 금액
        if not isinstance(amount, int) or isinstance(amount, bool):  # 정수만
            raise TypeError('할인 금액은 정수여야 합니다.')          # 자료형 틀림
        if amount < 0:                                               # 음수
            raise ValueError('할인 금액은 0 이상이어야 합니다.')     # 값 틀림
        self.amount = amount                                         # 저장

    def calculate(self, price):             # 오버라이딩
        return max(0, price - self.amount)  # 0원 아래로는 안 내려감

    def __str__(self):                    # 정책 이름
        return f'{self.amount:,}원 할인'  # 5000 → 5,000원


policies = [NoDiscount(), RateDiscount(0.1), AmountDiscount(5_000)]  # 종류가 다른 정책
for policy in policies:                                              # 종류를 묻지 않고
    print(f"{policy}: {policy.calculate(53_000):,}원")               # 같은 calculate() 호출
try:
    DiscountPolicy().calculate(1000)                                 # 부모를 그대로 사용
except NotImplementedError as e:                                     # 구현이 없음
    print("NotImplementedError:", e)                                 # 에러 문장

할인 없음: 53,000원
10% 할인: 47,700원
5,000원 할인: 48,000원
NotImplementedError: 자식 클래스에서 calculate()를 구현하세요.


In [12]:
# ---------- 2-2 ----------
class Customer:                                       # 고객: 장바구니를 가지고 정책을 받아 결제
    def __init__(self, name, cart=None):              # 장바구니는 선택
        self.name = name                              # 고객 이름
        self.cart = Cart() if cart is None else cart  # 없으면 고객마다 새 장바구니

    def add_to_cart(self, product):  # 상품 담기
        self.cart.add(product)       # 장바구니에 맡김 (위임)

    def checkout(self, discount_policy):                                  # 결제
        if not isinstance(discount_policy, DiscountPolicy):               # 정책 객체만 허용
            raise TypeError('DiscountPolicy 객체가 필요합니다.')          # 잘못된 인자
        if not self.cart:                                                 # Cart.__bool__ 사용
            return f'{self.name}: 장바구니가 비어 있습니다.'              # 빈 장바구니
        final_price = discount_policy.calculate(self.cart.total_price())  # 어떤 정책이든 같은 호출
        return f'{self.name} / {discount_policy} / 결제 금액 {final_price:,}원'  # 결과 문장


customer = Customer('민수', demo_cart)                           # 1-2의 장바구니를 가진 고객
for policy in policies:                                          # 2-1의 정책 세 개
    print(customer.checkout(policy))                             # 고객 코드는 하나
another_customer = Customer('지수')                              # 장바구니 없이 만든 고객
print("장바구니 공유?", customer.cart is another_customer.cart)  # 객체 독립성
print(another_customer.checkout(NoDiscount()))                   # 빈 장바구니

민수 / 할인 없음 / 결제 금액 53,000원
민수 / 10% 할인 / 결제 금액 47,700원
민수 / 5,000원 할인 / 결제 금액 48,000원
장바구니 공유? False
지수: 장바구니가 비어 있습니다.


In [13]:
# ---------- 3. 종합 실습 ----------
# 1단계: 잘못된 값은 객체가 스스로 막음 (원본 7절)
cases = [lambda: Product('잘못된 상품', -1),     # 음수 가격
         lambda: RateDiscount(1.5),              # 1을 넘는 할인율
         lambda: customer.checkout('10% 할인')]  # 정책 객체가 아닌 문자열
for make in cases:                               # 하나씩 실행
    try:
        make()                                   # 잘못된 값으로 만들거나 결제
    except (TypeError, ValueError) as error:     # 자료형 · 값 에러
        print(type(error).__name__, error)       # 에러 종류와 문장


# 2단계: 새 정책 — DiscountPolicy를 물려받아 calculate()만 채움
class CouponDiscount(DiscountPolicy):                # 일정 금액 이상이면 쿠폰 할인
    def __init__(self, minimum, amount):             # 최소 금액 · 할인 금액
        self.minimum, self.amount = minimum, amount  # 조건 저장

    def calculate(self, price):                                         # 오버라이딩
        return price - self.amount if price >= self.minimum else price  # 조건을 넘을 때만

    def __str__(self):                                            # 정책 이름
        return f'{self.minimum:,}원 이상 {self.amount:,}원 쿠폰'  # 출력용


# 3 · 4단계: Customer는 한 줄도 고치지 않고 새 정책으로 결제
coupon = CouponDiscount(50_000, 7_000)    # 5만 원 이상 7천 원 할인
print(customer.checkout(coupon))          # 53,000원 → 조건 충족
another_customer.add_to_cart(mouse)       # 지수는 마우스만
print(another_customer.checkout(coupon))  # 18,000원 → 조건 미달

ValueError 가격은 0 이상이어야 합니다.
ValueError 할인율은 0 이상 1 이하여야 합니다.
TypeError DiscountPolicy 객체가 필요합니다.
민수 / 50,000원 이상 7,000원 쿠폰 / 결제 금액 46,000원
지수 / 50,000원 이상 7,000원 쿠폰 / 결제 금액 18,000원
